# Module 7: Hardware Verification

**ACL2 for Computer Science — University-Level Tutorial Series**

## Learning Objectives

By the end of this module you will be able to:

1. Model basic digital logic gates as ACL2 functions
2. Build and verify combinational circuits (adders)
3. Represent multi-bit values as boolean vectors and reason about them
4. Model sequential circuits with state
5. Appreciate how these techniques scale to real processor verification (FM9001)

## 7.1 Introduction — Why Hardware Verification?

Hardware verification is one of ACL2's **flagship applications**. The most celebrated result is the **FM9001**: a complete microprocessor whose design was formally verified end-to-end in ACL2 by Warren Hunt and colleagues.

Why does hardware need formal verification?

- A single bug in a processor can cost billions (recall the Intel FDIV bug, ~\$475 million)
- Hardware cannot be "patched" after fabrication
- Exhaustive testing is infeasible for circuits with $2^{64}$ or more input combinations

Digital circuits can be modeled as **boolean functions**: they take boolean inputs and produce boolean outputs. This makes them a natural fit for theorem proving.

### The Verification Approach

The basic methodology is:

1. **Model** the hardware design as ACL2 functions
2. **Specify** the intended behavior as a separate ACL2 function
3. **Prove** that the model implements the specification

We start with the simplest building blocks — logic gates — and work our way up to multi-bit circuits.

## 7.2 Boolean Gates in ACL2

In ACL2, we represent boolean values as `t` (true) and `nil` (false). We can model the fundamental logic gates using nested `if` expressions.

Recall that `(if test then else)` returns `then` when `test` is non-nil, and `else` when `test` is nil.

In [ ]:
; Boolean AND gate
(defun b-and (a b)
  (if a (if b t nil) nil))

In [ ]:
; Boolean OR gate
(defun b-or (a b)
  (if a t (if b t nil)))

In [ ]:
; Boolean NOT gate
(defun b-not (a)
  (if a nil t))

In [ ]:
; Boolean XOR gate
(defun b-xor (a b)
  (if a (if b nil t) (if b t nil)))

In [ ]:
; Boolean NAND gate
(defun b-nand (a b)
  (b-not (b-and a b)))

In [ ]:
; Boolean NOR gate
(defun b-nor (a b)
  (b-not (b-or a b)))

### Testing Our Gates

Let's verify our gate definitions match the expected truth tables:

In [ ]:
; AND truth table
(list (b-and nil nil)   ; nil
      (b-and nil t)     ; nil
      (b-and t nil)     ; nil
      (b-and t t))      ; t

In [ ]:
; XOR truth table
(list (b-xor nil nil)   ; nil
      (b-xor nil t)     ; t
      (b-xor t nil)     ; t
      (b-xor t t))      ; nil

### De Morgan's Laws

De Morgan's laws are fundamental identities in boolean algebra:

$$\neg(A \wedge B) = \neg A \vee \neg B$$
$$\neg(A \vee B) = \neg A \wedge \neg B$$

Let's prove these for our gate definitions. Since our inputs are booleans (either `t` or `nil`), ACL2 can verify these by case analysis.

In [ ]:
; De Morgan's first law: NOT(A AND B) = (NOT A) OR (NOT B)
(thm
  (equal (b-not (b-and a b))
         (b-or (b-not a) (b-not b))))

In [ ]:
; De Morgan's second law: NOT(A OR B) = (NOT A) AND (NOT B)
(thm
  (equal (b-not (b-or a b))
         (b-and (b-not a) (b-not b))))

In [ ]:
; XOR self-inverse property: a XOR a = false
(thm
  (equal (b-xor a a) nil))

In [ ]:
; XOR with nil is identity
(thm
  (equal (b-xor a nil) (if a t nil)))

In [ ]:
; NAND is universal: we can build NOT from NAND
(thm
  (equal (b-nand a a)
         (b-not a)))

## 7.3 Combinational Circuits — The Full Adder

A **combinational circuit** computes outputs purely from its current inputs (no memory or state). The most fundamental arithmetic circuit is the **full adder**.

A 1-bit full adder takes three inputs — `a`, `b`, and `cin` (carry-in) — and produces two outputs:
- **sum**: the low bit of $a + b + c_{in}$
- **carry**: the high bit of $a + b + c_{in}$

$$\text{sum} = a \oplus b \oplus c_{in}$$
$$\text{carry} = (a \wedge b) \vee (c_{in} \wedge (a \oplus b))$$

In [ ]:
; Full adder: sum output
(defun full-adder-sum (a b cin)
  (b-xor (b-xor a b) cin))

In [ ]:
; Full adder: carry output
(defun full-adder-carry (a b cin)
  (b-or (b-and a b)
        (b-and cin (b-xor a b))))

### Correctness of the Full Adder

To state correctness, we need a way to convert booleans to numbers. We define a helper that maps `t` to 1 and `nil` to 0:

In [ ]:
; Convert a boolean to a natural number
(defun bool-to-nat (x)
  (if x 1 0))

In [ ]:
; The full adder is correct: the two-bit output (carry, sum)
; represents the arithmetic sum a + b + cin.
(thm
  (equal (+ (* 2 (bool-to-nat (full-adder-carry a b cin)))
            (bool-to-nat (full-adder-sum a b cin)))
         (+ (bool-to-nat a)
            (bool-to-nat b)
            (bool-to-nat cin))))

This theorem states that the numeric value of the two-bit result (carry as the high bit, sum as the low bit) equals the arithmetic sum of the three input bits. ACL2 proves this automatically by case analysis.

### Ripple-Carry Adder

We can chain full adders to add multi-bit numbers. Each adder's carry-out feeds into the next adder's carry-in, forming a **ripple-carry adder**.

We represent multi-bit numbers as lists of booleans (least-significant bit first).

In [ ]:
; Ripple-carry adder: adds two bit-vectors with a carry-in.
; Returns a list of sum bits (least-significant first).
(defun ripple-carry-adder (x y cin)
  (if (and (endp x) (endp y))
      (if cin (list t) nil)
    (let* ((a (if (consp x) (car x) nil))
           (b (if (consp y) (car y) nil))
           (sum-bit (full-adder-sum a b cin))
           (carry-out (full-adder-carry a b cin)))
      (cons sum-bit
            (ripple-carry-adder (cdr x) (cdr y) carry-out)))))

In [ ]:
; Test: 3 + 5 = 8 in 4-bit binary
; 3 = (t t nil nil), 5 = (t nil t nil)
; 8 = (nil nil nil t)
(ripple-carry-adder (list t t nil nil)
                    (list t nil t nil)
                    nil)

## 7.4 Bit Vectors

To reason about multi-bit hardware, we represent values as **bit vectors** — lists of booleans with the least-significant bit first.

We need conversion functions between bit vectors and natural numbers:

In [ ]:
; Convert a bit vector (LSB first) to a natural number
(defun bv-to-nat (bv)
  (if (endp bv)
      0
    (+ (if (car bv) 1 0)
       (* 2 (bv-to-nat (cdr bv))))))

In [ ]:
; Convert a natural number to a bit vector of width w
(defun nat-to-bv (n w)
  (if (zp w)
      nil
    (cons (not (equal (mod n 2) 0))
          (nat-to-bv (floor n 2) (- w 1)))))

In [ ]:
; Test conversions
(list (bv-to-nat (list t t nil nil))   ; 3
      (bv-to-nat (list t nil t nil))   ; 5
      (nat-to-bv 13 4))                ; (t nil t t) = 13

In [ ]:
; Round-trip property: converting a small number to bv and back
(thm
  (implies (and (natp n)
                (natp w)
                (< n (expt 2 w)))
           (equal (bv-to-nat (nat-to-bv n w)) n)))

### Bit-Vector Addition

We define addition on bit vectors using our ripple-carry adder:

In [ ]:
; Bit-vector addition
(defun bv-add (x y)
  (ripple-carry-adder x y nil))

In [ ]:
; Test: 3 + 5 = 8
(bv-to-nat (bv-add (nat-to-bv 3 4)
                   (nat-to-bv 5 4)))

### Correctness of Bit-Vector Addition

The key correctness theorem for our adder states that bit-vector addition correctly implements modular arithmetic:

$$\text{bv-to-nat}(\text{bv-add}(x, y)) = (\text{bv-to-nat}(x) + \text{bv-to-nat}(y))$$

Note that `bv-add` may produce an output one bit wider than the inputs (due to carry-out), so the full sum is captured without modular truncation.

In [ ]:
; Adder correctness: the adder computes the arithmetic sum.
; (The ripple-carry adder preserves the full sum including carry.)
(defthm ripple-carry-adder-correct
  (equal (bv-to-nat (ripple-carry-adder x y cin))
         (+ (bv-to-nat x)
            (bv-to-nat y)
            (bool-to-nat cin))))

## 7.5 Sequential Circuits

Unlike combinational circuits, **sequential circuits** have memory — their outputs depend on both current inputs and stored state. The fundamental storage element is the **flip-flop**, which captures a value on each clock tick.

We model sequential circuits by making state explicit: a circuit is a function from `(state, input) → (new-state, output)`.

In [ ]:
; A simple 4-bit counter circuit.
; State is a 4-bit number (0..15), input is an enable signal.
; When enabled, the counter increments (mod 16).
(defun counter-next (state enable)
  (if enable
      (mod (+ state 1) 16)
    state))

In [ ]:
; Run the counter for n clock cycles, all enabled
(defun run-counter (state n)
  (if (zp n)
      state
    (run-counter (counter-next state t) (- n 1))))

In [ ]:
; The counter counts correctly: after n steps from 0,
; the state is n mod 16.
(defthm counter-correct
  (implies (and (natp n)
                (natp state)
                (< state 16))
           (equal (run-counter state n)
                  (mod (+ state n) 16))))

### State Machine Example

Let's model a simple state machine — a traffic light controller with states `green`, `yellow`, and `red`:

In [ ]:
; Traffic light state machine
(defun next-light (state)
  (cond ((equal state 'green)  'yellow)
        ((equal state 'yellow) 'red)
        ((equal state 'red)    'green)
        (t                     'red)))  ; default

In [ ]:
; Run the traffic light for n cycles
(defun run-light (state n)
  (if (zp n)
      state
    (run-light (next-light state) (- n 1))))

In [ ]:
; After 3 cycles, we return to the same state
(thm
  (implies (member-equal state '(green yellow red))
           (equal (run-light state 3) state)))

## 7.6 The FM9001 Story

The **FM9001** is a 32-bit microprocessor whose gate-level design was **fully verified** in ACL2 by Warren A. Hunt, Jr. This is one of the landmark achievements in formal verification.

### What Was Proved

The FM9001 verification established that:

- The **gate-level netlist** (the actual hardware design) correctly implements
- A **high-level instruction set specification**
- For **all possible inputs and initial states**

This means every instruction, every addressing mode, and every edge case was formally verified — not tested, but *proved*.

### Scale of the Proof

| Aspect | Detail |
|--------|--------|
| Processor | 32-bit, custom ISA |
| Gate count | ~10,000 gates |
| Lines of ACL2 | ~20,000 |
| Properties proved | Complete functional correctness |
| Years of effort | ~3 years |

### Where to Find It

The FM9001 proof is available in the ACL2 Community Books:

```
books/projects/fm9001/
```

Key files include the netlist description, the ISA specification, and the main correctness theorems linking them.

### Significance

The FM9001 demonstrated that:

1. **Formal verification of processors is feasible** — you can prove a real processor correct
2. **The ACL2 approach scales** — from single gates to complete processors
3. **Hierarchical verification works** — verify gates → modules → processor

This work influenced industrial practice: similar techniques are now used at AMD, Intel, ARM, and other chip companies for verifying critical components.

Other notable ACL2 hardware verification projects include:
- **Centaur Technology** (now AMD): Verified floating-point units in production processors
- **VIA processor verification**: Using ACL2 for commercial x86 verification
- The `books/centaur/` directory contains many industrially-relevant hardware verification tools

## 7.7 Exercises

### Exercise 7.1: NAND-Only Circuits

NAND is a **universal gate** — any boolean function can be built from NAND alone.

Define `b-or-from-nand` that implements OR using only `b-nand` (and `b-not` built from NAND). Then prove it is equivalent to `b-or`.

In [ ]:
; Exercise 7.1: Define OR using only NAND gates
; Hint: OR(a,b) = NAND(NOT(a), NOT(b)) = NAND(NAND(a,a), NAND(b,b))

(defun b-or-from-nand (a b)
  ;; YOUR DEFINITION HERE
  nil)

; Prove equivalence:
; (thm (equal (b-or-from-nand a b) (b-or a b)))

### Exercise 7.2: 2-to-1 Multiplexer

A **multiplexer** (mux) selects between two inputs based on a select signal:
- When `sel` is nil, output `a`
- When `sel` is t, output `b`

Define `mux2` using basic gates, and prove it correct.

In [ ]:
; Exercise 7.2: Define a 2-to-1 multiplexer from basic gates

(defun mux2 (sel a b)
  ;; YOUR DEFINITION HERE
  nil)

; Prove:
; (thm (equal (mux2 nil a b) (if a t nil)))
; (thm (equal (mux2 t a b) (if b t nil)))

### Exercise 7.3: Subtraction Circuit

A subtractor can be built from an adder by using two's complement: $a - b = a + \overline{b} + 1$.

Define `bv-not` that inverts every bit in a bit vector, then define `bv-sub` using `bv-not` and your ripple-carry adder with carry-in set to `t`.

In [ ]:
; Exercise 7.3: Bit-vector subtraction

(defun bv-not (x)
  ;; YOUR DEFINITION HERE: invert each bit
  nil)

(defun bv-sub (x y)
  ;; YOUR DEFINITION HERE: use ripple-carry-adder with inverted y and cin=t
  nil)

; Test: 7 - 3 = 4 in 4-bit
; (bv-to-nat (bv-sub (nat-to-bv 7 4) (nat-to-bv 3 4)))

### Exercise 7.4: Equality Comparator

Define `bv-equal` that takes two bit vectors and returns `t` if they represent the same value and `nil` otherwise. Prove that equal bit vectors give `t` and that the function is reflexive.

In [ ]:
; Exercise 7.4: Bit-vector equality comparator

(defun bv-equal (x y)
  ;; YOUR DEFINITION HERE
  nil)

; Prove reflexivity:
; (thm (equal (bv-equal x x) t))

---

**Navigation:**
[< Module 6 — Machine Models](06_machine_models.ipynb) | [Module 8 — SAT Solving and Boolean Reasoning >](08_sat_boolean.ipynb)